In [1]:
# load libraries and input all data 

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""      # Force CPU

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

DEVICE = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)


sig_df = pd.read_csv('data/CoLBTHydro.csv', header=None)
bkg_df = pd.read_csv('data/Jewel.csv', header=None)

# Data structure:
# Column 0:     n_objects (integer)
# Columns 1-7:  Event-level features (7 values)
# Columns 8+:   Per-object features in groups of 4: (pT, eta, phi, flag)

MAX_PARTICLES = 50             # Maximum particles to keep per event
N_FEAT_PER_PARTICLE = 4        # pT, eta, phi, flag
EVENT_FEAT_COLS = slice(1, 8)  # Columns 1 through 7
OBJ_START = 8                  # Per-particle data starts here


def parse_data(df, max_particles=MAX_PARTICLES):
    """
    Parse raw CSV into structured arrays.
    
    Returns:
    --------
    event_features : (n_events, 7) — global event properties
    particle_features : (n_events, max_particles, 4) — per-particle (pT, eta, phi, flag)
    particle_mask : (n_events, max_particles) — 1 where real particle, 0 where padded
    n_particles : (n_events,) — actual number of particles per event
    """
    raw = df.values
    n_events = len(df)
    
    # Event-level features
    event_features = raw[:, EVENT_FEAT_COLS].astype(np.float32)
    
    # Per-particle features
    particle_features = np.zeros((n_events, max_particles, N_FEAT_PER_PARTICLE), dtype=np.float32)
    particle_mask = np.zeros((n_events, max_particles), dtype=np.float32)
    n_particles = np.zeros(n_events, dtype=np.int32)
    
    for i in range(n_events):
        n_obj = int(raw[i, 0])                         # Number of particles in event
        n_obj = min(n_obj, max_particles)              # Cap at maximum
        n_particles[i] = n_obj
        
        for j in range(n_obj):
            col_start = OBJ_START + j * N_FEAT_PER_PARTICLE
            col_end = col_start + N_FEAT_PER_PARTICLE
            if col_end <= raw.shape[1]:
                particle_features[i, j, :] = raw[i, col_start:col_end]
                particle_mask[i, j] = 1.0              # Mark as real particle
    
    return event_features, particle_features, particle_mask, n_particles


# Parse signal and background
sig_evt, sig_parts, sig_mask, sig_nparts = parse_data(sig_df)
bkg_evt, bkg_parts, bkg_mask, bkg_nparts = parse_data(bkg_df)

# Combine
event_features = np.concatenate([sig_evt, bkg_evt], axis=0)
particle_features = np.concatenate([sig_parts, bkg_parts], axis=0)
particle_mask = np.concatenate([sig_mask, bkg_mask], axis=0)
labels = np.concatenate([np.ones(len(sig_evt)), np.zeros(len(bkg_evt))]).astype(np.float32)

print(f"Total events: {len(labels)}")
print(f"Event features shape: {event_features.shape}")      # (N, 7)
print(f"Particle features shape: {particle_features.shape}")  # (N, 50, 4)
print(f"Particle mask shape: {particle_mask.shape}")          # (N, 50)

# Train/test split
(evt_train, evt_test, 
 parts_train, parts_test, 
 mask_train, mask_test, 
 y_train, y_test) = train_test_split(
    event_features, particle_features, particle_mask, labels,
    test_size=0.2, random_state=42, stratify=labels
)

print(f"\nTrain: {len(y_train)}, Test: {len(y_test)}")
print(f"Signal (train): {y_train.sum():.0f}, Background (train): {len(y_train)-y_train.sum():.0f}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/opt/anaconda3/envs/work_py_ml/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 807, in start
    self.io_loop.sta

Total events: 155025
Event features shape: (155025, 7)
Particle features shape: (155025, 50, 4)
Particle mask shape: (155025, 50)

Train: 124020, Test: 31005
Signal (train): 36166, Background (train): 87854


In [2]:
# this is a bunch of useful shared info for the dataset 

class EventDataset(Dataset):
    """Custom dataset that holds event features, particle features, mask, and labels."""
    def __init__(self, evt_feats, part_feats, mask, labels):
        self.evt = torch.FloatTensor(evt_feats)
        self.parts = torch.FloatTensor(part_feats)
        self.mask = torch.FloatTensor(mask)
        self.labels = torch.FloatTensor(labels).unsqueeze(1)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.evt[idx], self.parts[idx], self.mask[idx], self.labels[idx]


def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, patience=10):
    """
    Generic training loop with early stopping.
    
    Parameters:
    -----------
    model : nn.Module
    train_loader : DataLoader
    val_loader : DataLoader
    epochs : int
    lr : float
    patience : int — early stopping patience
    
    Returns:
    --------
    dict with training history
    """
    criterion = nn.BCELoss()                             # Binary cross-entropy
    optimizer = optim.Adam(model.parameters(), lr=lr)    # Adam optimizer
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    
    for epoch in range(epochs):
        # --- Training ---
        model.train()
        train_loss = 0.0
        for evt, parts, mask, labels in train_loader:
            optimizer.zero_grad()
            output = model(evt, parts, mask)             # Forward pass
            loss = criterion(output, labels)
            loss.backward()                              # Backprop
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
            optimizer.step()
            train_loss += loss.item() * evt.size(0)
        train_loss /= len(train_loader.dataset)
        
        # --- Validation ---
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_labels = []
        with torch.no_grad():
            for evt, parts, mask, labels in val_loader:
                output = model(evt, parts, mask)
                loss = criterion(output, labels)
                val_loss += loss.item() * evt.size(0)
                val_preds.append(output.numpy())
                val_labels.append(labels.numpy())
        val_loss /= len(val_loader.dataset)
        val_preds = np.concatenate(val_preds).ravel()
        val_labels = np.concatenate(val_labels).ravel()
        val_auc = roc_auc_score(val_labels, val_preds)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}: train_loss={train_loss:.4f}, "
                  f"val_loss={val_loss:.4f}, val_auc={val_auc:.4f}")
    
    # Restore best weights
    model.load_state_dict(best_state)
    return history


def evaluate_model(model, test_loader, y_test, model_name):
    """Evaluate model and return predictions."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for evt, parts, mask, labels in test_loader:
            output = model(evt, parts, mask)
            all_preds.append(output.numpy())
    
    y_prob = np.concatenate(all_preds).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    print(f"\n{model_name} Results:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  AUC-ROC:  {auc:.4f}")
    
    return y_prob, acc, auc


# --- Create DataLoaders ---
# Split train into train/val
evt_tr, evt_val, parts_tr, parts_val, mask_tr, mask_val, y_tr, y_val = train_test_split(
    evt_train, parts_train, mask_train, y_train, test_size=0.15, random_state=42
)

train_dataset = EventDataset(evt_tr, parts_tr, mask_tr, y_tr)
val_dataset = EventDataset(evt_val, parts_val, mask_val, y_val)
test_dataset = EventDataset(evt_test, parts_test, mask_test, y_test)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

Train batches: 1648, Val batches: 291, Test batches: 485


In [3]:
# ============================================================
# MODEL 1: PARTICLE FLOW NETWORK (PFN)
# ============================================================
# Reference: arXiv:1810.05165 (Komiske, Metodiev, Thaler 2019)
#
# Architecture:
#   Per-particle: Φ(pT_i, η_i, φ_i, flag_i) → latent z_i
#   Aggregate:    Z = Σ z_i (sum over particles, masked)
#   Classify:     F(Z, event_features) → signal probability
#
# Key property: Permutation invariant by construction (sum pooling)

print("=" * 60)
print("MODEL 1: PARTICLE FLOW NETWORK (PFN)")
print("=" * 60)


class PFN(nn.Module):
    """
    Particle Flow Network.
    
    Φ-network maps each particle to a latent space.
    Sum pooling aggregates over all particles (respecting mask).
    F-network classifies the aggregated representation.
    """
    def __init__(self, particle_dims=4, event_dims=7, 
                 phi_layers=[64, 64, 128], f_layers=[128, 64, 32]):
        """
        Parameters:
        -----------
        particle_dims : int
            Number of features per particle (pT, eta, phi, flag = 4).
        event_dims : int
            Number of event-level features (7).
        phi_layers : list of int
            Hidden layer sizes for per-particle Φ-network.
        f_layers : list of int
            Hidden layer sizes for classification F-network.
        """
        super(PFN, self).__init__()
        
        # Φ-network: per-particle mapping
        # Maps (pT, eta, phi, flag) → latent vector
        phi_modules = []
        in_dim = particle_dims
        for out_dim in phi_layers:
            phi_modules.append(nn.Linear(in_dim, out_dim))   # Linear transformation
            phi_modules.append(nn.ReLU())                    # Non-linearity
            in_dim = out_dim
        self.phi_net = nn.Sequential(*phi_modules)
        
        # F-network: event-level classification
        # Input: aggregated latent (phi_layers[-1]) + event features (event_dims)
        f_modules = []
        in_dim = phi_layers[-1] + event_dims                 # Concatenate particle + event info
        for out_dim in f_layers:
            f_modules.append(nn.Linear(in_dim, out_dim))
            f_modules.append(nn.ReLU())
            f_modules.append(nn.Dropout(0.1))
            in_dim = out_dim
        f_modules.append(nn.Linear(in_dim, 1))              # Final output
        f_modules.append(nn.Sigmoid())                       # Probability
        self.f_net = nn.Sequential(*f_modules)
    
    def forward(self, event_feats, particle_feats, mask):
        """
        Parameters:
        -----------
        event_feats : (batch, 7) — event-level features
        particle_feats : (batch, max_particles, 4) — per-particle features
        mask : (batch, max_particles) — 1 for real particles, 0 for padding
        
        Returns:
        --------
        (batch, 1) — signal probability
        """
        # Apply Φ-network to each particle: (batch, N, 4) → (batch, N, latent_dim)
        phi_output = self.phi_net(particle_feats)
        
        # Apply mask: zero out padded particles before summing
        # mask shape: (batch, N) → (batch, N, 1) for broadcasting
        phi_output = phi_output * mask.unsqueeze(-1)
        
        # Sum pooling over particles: (batch, N, latent_dim) → (batch, latent_dim)
        # This makes the model permutation-invariant
        aggregated = phi_output.sum(dim=1)
        
        # Concatenate with event-level features
        combined = torch.cat([aggregated, event_feats], dim=1)
        
        # Classify
        output = self.f_net(combined)
        return output


# Instantiate
pfn_model = PFN(
    particle_dims=N_FEAT_PER_PARTICLE,       # 4
    event_dims=7,                            # 7 event features
    phi_layers=[64, 64, 128],                # Φ-network layers
    f_layers=[128, 64, 32]                   # F-network layers
)
print(f"PFN parameters: {sum(p.numel() for p in pfn_model.parameters()):,}")

# Train
history_pfn = train_model(pfn_model, train_loader, val_loader, epochs=50, lr=0.001)

# Evaluate
y_prob_pfn, acc_pfn, auc_pfn = evaluate_model(pfn_model, test_loader, y_test, "PFN")

MODEL 1: PARTICLE FLOW NETWORK (PFN)
PFN parameters: 40,577


RuntimeError: Numpy is not available

In [4]:
import numpy as np
import torch
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

NumPy: 2.4.6
PyTorch: 1.13.1


In [ ]:
# ============================================================
# MODEL 2: DEEP SETS
# ============================================================
# Reference: arXiv:1703.06114 (Zaheer et al., 2017)
#
# Similar to PFN but with both sum AND max pooling,
# plus a more expressive per-particle network.
#
# Key difference from PFN: multiple aggregation functions

print("\n" + "=" * 60)
print("MODEL 2: DEEP SETS")
print("=" * 60)


class DeepSets(nn.Module):
    """
    Deep Sets model with multiple pooling operations.
    
    Uses both sum and max pooling for richer event representation.
    """
    def __init__(self, particle_dims=4, event_dims=7,
                 encoder_layers=[64, 128, 128], 
                 decoder_layers=[256, 128, 64]):
        super(DeepSets, self).__init__()
        
        # Per-particle encoder (equivariant part)
        enc_modules = []
        in_dim = particle_dims
        for out_dim in encoder_layers:
            enc_modules.append(nn.Linear(in_dim, out_dim))
            enc_modules.append(nn.BatchNorm1d(out_dim))      # Batch norm per feature
            enc_modules.append(nn.ReLU())
            in_dim = out_dim
        self.encoder = nn.ModuleList([
            nn.Linear(particle_dims, encoder_layers[0]),
            nn.ReLU(),
            nn.Linear(encoder_layers[0], encoder_layers[1]),
            nn.ReLU(),
            nn.Linear(encoder_layers[1], encoder_layers[2]),
            nn.ReLU()
        ])
        
        # Simpler sequential encoder
        self.particle_encoder = nn.Sequential(
            nn.Linear(particle_dims, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        # Decoder (invariant part)
        # Input: sum_pool (128) + max_pool (128) + mean_pool (128) + event_feats (7)
        latent_dim = encoder_layers[-1]
        decoder_input_dim = latent_dim * 3 + event_dims      # 3 pooling ops + event features
        
        self.decoder = nn.Sequential(
            nn.Linear(decoder_input_dim, decoder_layers[0]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(decoder_layers[0], decoder_layers[1]),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(decoder_layers[1], decoder_layers[2]),
            nn.ReLU(),
            nn.Linear(decoder_layers[2], 1),
            nn.Sigmoid()
        )
    
    def forward(self, event_feats, particle_feats, mask):
        """
        event_feats: (batch, 7)
        particle_feats: (batch, N, 4)
        mask: (batch, N)
        """
        # Encode each particle: (batch, N, 4) → (batch, N, 128)
        encoded = self.particle_encoder(particle_feats)
        
        # Apply mask
        mask_expanded = mask.unsqueeze(-1)                    # (batch, N, 1)
        encoded_masked = encoded * mask_expanded              # Zero out padding
        
        # Sum pooling: sum over particles
        sum_pool = encoded_masked.sum(dim=1)                 # (batch, 128)
        
        # Max pooling: max over particles (set padded to -inf before max)
        encoded_for_max = encoded_masked + (1 - mask_expanded) * (-1e9)
        max_pool = encoded_for_max.max(dim=1)[0]             # (batch, 128)
        
        # Mean pooling: sum / count of real particles
        n_particles = mask.sum(dim=1, keepdim=True).clamp(min=1)  # Avoid div by 0
        mean_pool = sum_pool / n_particles                   # (batch, 128)
        
        # Concatenate all representations
        combined = torch.cat([sum_pool, max_pool, mean_pool, event_feats], dim=1)
        
        # Decode
        output = self.decoder(combined)
        return output


# Instantiate
deepsets_model = DeepSets(
    particle_dims=N_FEAT_PER_PARTICLE,
    event_dims=7,
    encoder_layers=[64, 128, 128],
    decoder_layers=[256, 128, 64]
)
print(f"Deep Sets parameters: {sum(p.numel() for p in deepsets_model.parameters()):,}")

# Train
history_ds = train_model(deepsets_model, train_loader, val_loader, epochs=50, lr=0.001)

# Evaluate
y_prob_ds, acc_ds, auc_ds = evaluate_model(deepsets_model, test_loader, y_test, "Deep Sets")

In [ ]:
# ============================================================
# MODEL 3: PARTICLENET (Dynamic Graph CNN with EdgeConv)
# ============================================================
# Reference: arXiv:1902.08570 (Qu & Gouskos, 2020)
#
# Architecture:
#   1. Build k-nearest-neighbor graph in feature space
#   2. Apply EdgeConv: edge features = [h_i, h_j - h_i] → MLP → aggregate
#   3. Dynamically rebuild graph after each layer (in updated feature space)
#   4. Global pooling → classifier
#
# Key: learns particle relationships through graph structure

print("\n" + "=" * 60)
print("MODEL 3: PARTICLENET (EdgeConv GNN)")
print("=" * 60)


def knn(x, k, mask=None):
    """
    Compute k-nearest neighbors based on Euclidean distance.
    
    Parameters:
    -----------
    x : (batch, N, C) — point features
    k : int — number of neighbors
    mask : (batch, N) — valid particle mask
    
    Returns:
    --------
    idx : (batch, N, k) — indices of k nearest neighbors for each point
    """
    # Pairwise distance: ||x_i - x_j||^2
    # = ||x_i||^2 + ||x_j||^2 - 2 * x_i · x_j
    inner = torch.bmm(x, x.transpose(1, 2))             # (batch, N, N)
    xx = (x * x).sum(dim=-1, keepdim=True)               # (batch, N, 1)
    dist = xx + xx.transpose(1, 2) - 2 * inner           # (batch, N, N)
    
    # Mask out padded particles (set distance to infinity)
    if mask is not None:
        # mask: (batch, N) → expand for pairwise
        mask_matrix = mask.unsqueeze(1)                   # (batch, 1, N)
        dist = dist + (1 - mask_matrix) * 1e9            # Padded → far away
    
    # Get k nearest neighbors (smallest distances, excluding self)
    dist[:, torch.arange(x.size(1)), torch.arange(x.size(1))] = 1e9  # Exclude self
    _, idx = dist.topk(k, dim=-1, largest=False)         # (batch, N, k) smallest
    
    return idx


class EdgeConvBlock(nn.Module):
    """
    EdgeConv layer from DGCNN/ParticleNet.
    
    For each point i and its neighbor j:
        edge_feature = [h_i, h_j - h_i]   (concatenation)
        → MLP → aggregate (max over neighbors)
    """
    def __init__(self, in_channels, out_channels, k=16):
        """
        Parameters:
        -----------
        in_channels : int
            Input feature dimension per point.
        out_channels : list of int
            Output channels for the edge MLP.
        k : int
            Number of nearest neighbors.
        """
        super(EdgeConvBlock, self).__init__()
        self.k = k
        
        # Edge MLP: input is [h_i || h_j - h_i] so input_dim = 2 * in_channels
        edge_layers = []
        in_dim = 2 * in_channels
        for out_dim in out_channels:
            edge_layers.append(nn.Linear(in_dim, out_dim))
            edge_layers.append(nn.BatchNorm1d(out_dim))
            edge_layers.append(nn.ReLU())
            in_dim = out_dim
        self.edge_mlp = nn.Sequential(*edge_layers)
        
        # Shortcut connection if dimensions change
        self.shortcut = nn.Sequential(
            nn.Linear(in_channels, out_channels[-1]),
            nn.BatchNorm1d(out_channels[-1])
        ) if in_channels != out_channels[-1] else nn.Identity()
    
    def forward(self, x, mask=None):
        """
        x : (batch, N, C) — point features
        mask : (batch, N) — particle mask
        
        Returns:
        --------
        (batch, N, out_channels[-1])
        """
        batch_size, n_points, n_features = x.shape
        
        # Find k nearest neighbors
        idx = knn(x, self.k, mask)                       # (batch, N, k)
        
        # Gather neighbor features
        # idx: (batch, N, k) → expand to (batch, N, k, C)
        idx_expanded = idx.unsqueeze(-1).expand(-1, -1, -1, n_features)
        x_expanded = x.unsqueeze(2).expand(-1, -1, self.k, -1)   # (batch, N, k, C)
        neighbors = torch.gather(
            x.unsqueeze(1).expand(-1, n_points, -1, -1),          # (batch, N, N, C)
            2, idx_expanded                                        # gather along N dim
        )                                                          # (batch, N, k, C)
        
        # Edge features: [h_i, h_j - h_i]
        x_repeated = x.unsqueeze(2).expand_as(neighbors)          # (batch, N, k, C)
        edge_features = torch.cat([x_repeated, neighbors - x_repeated], dim=-1)  # (B, N, k, 2C)
        
        # Apply MLP to edges
        # Reshape for BatchNorm: (B*N*k, 2C)
        edge_flat = edge_features.reshape(-1, 2 * n_features)
        
        # Apply edge MLP layer by layer (handling BatchNorm)
        out = edge_flat
        for layer in self.edge_mlp:
            if isinstance(layer, nn.BatchNorm1d):
                out = layer(out)
            else:
                out = layer(out)
        
        # Reshape back: (batch, N, k, out_dim)
        out_dim = out.shape[-1]
        out = out.reshape(batch_size, n_points, self.k, out_dim)
        
        # Aggregate: max pooling over neighbors
        out = out.max(dim=2)[0]                          # (batch, N, out_dim)
        
        # Shortcut connection
        shortcut = x.reshape(-1, n_features)
        shortcut = self.shortcut(shortcut).reshape(batch_size, n_points, -1)
        
        out = out + shortcut                             # Residual connection
        
        # Apply mask
        if mask is not None:
            out = out * mask.unsqueeze(-1)
        
        return out


class ParticleNet(nn.Module):
    """
    ParticleNet: Dynamic Graph CNN for particle classification.
    
    Stacks multiple EdgeConv blocks, rebuilding the graph each time
    in the updated feature space.
    """
    def __init__(self, particle_dims=4, event_dims=7, k=7,
                 conv_channels=[[32, 32, 32], [64, 64, 64], [128, 128, 128]]):
        """
        Parameters:
        -----------
        particle_dims : int
            Input features per particle.
        event_dims : int
            Event-level features.
        k : int
            Number of nearest neighbors.
        conv_channels : list of list of int
            Output channels for each EdgeConv block.
        """
        super(ParticleNet, self).__init__()
        
        # Initial projection
        self.input_proj = nn.Sequential(
            nn.Linear(particle_dims, 32),
            nn.ReLU()
        )
        
        # EdgeConv blocks
        self.edge_convs = nn.ModuleList()
        in_ch = 32
        for channels in conv_channels:
            self.edge_convs.append(EdgeConvBlock(in_ch, channels, k=k))
            in_ch = channels[-1]
        
        # Global pooling + classifier
        pool_dim = in_ch                                 # Last EdgeConv output dim
        self.classifier = nn.Sequential(
            nn.Linear(pool_dim + event_dims, 128),       # Pool + event features
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, event_feats, particle_feats, mask):
        """
        event_feats: (batch, 7)
        particle_feats: (batch, N, 4)
        mask: (batch, N)
        """
        # Project input particles
        x = self.input_proj(particle_feats)              # (batch, N, 32)
        x = x * mask.unsqueeze(-1)                       # Mask padding
        
        # Apply EdgeConv blocks (graph is rebuilt each time in new feature space)
        for edge_conv in self.edge_convs:
            x = edge_conv(x, mask)                       # (batch, N, C)
        
        # Global average pooling (over particles, masked)
        x_masked = x * mask.unsqueeze(-1)
        n_particles = mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = x_masked.sum(dim=1) / n_particles       # (batch, C)
        
        # Combine with event features and classify
        combined = torch.cat([pooled, event_feats], dim=1)
        output = self.classifier(combined)
        return output


# Instantiate (use smaller k for CPU speed)
particlenet_model = ParticleNet(
    particle_dims=N_FEAT_PER_PARTICLE,
    event_dims=7,
    k=7,                                                 # 7 neighbors (smaller for CPU)
    conv_channels=[[32, 32, 32], [64, 64, 64]]           # 2 blocks (lighter for CPU)
)
print(f"ParticleNet parameters: {sum(p.numel() for p in particlenet_model.parameters()):,}")

# Train
history_pnet = train_model(particlenet_model, train_loader, val_loader, epochs=50, lr=0.001)

# Evaluate
y_prob_pnet, acc_pnet, auc_pnet = evaluate_model(
    particlenet_model, test_loader, y_test, "ParticleNet"
)

In [ ]:
# ============================================================
# MODEL 4: PARTICLE TRANSFORMER (ParT)
# ============================================================
# Reference: arXiv:2202.03772 (Qu et al., 2022)
#
# Architecture:
#   - Self-attention over all particle pairs
#   - Pairwise interaction features injected into attention weights
#   - Class token for classification (or mean pooling)
#
# Key innovation: pairwise features (ΔR, ΔpT, etc.) modify attention scores

print("\n" + "=" * 60)
print("MODEL 4: PARTICLE TRANSFORMER (ParT)")
print("=" * 60)


class PairwiseFeatures(nn.Module):
    """
    Compute pairwise interaction features between all particle pairs.
    These are used as bias in the attention mechanism.
    """
    def __init__(self, pair_feat_dim, d_model):
        super(PairwiseFeatures, self).__init__()
        # Project pairwise features to attention bias
        self.pair_proj = nn.Sequential(
            nn.Linear(pair_feat_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1)                        # Scalar bias per head
        )
    
    def forward(self, particle_feats, mask):
        """
        Compute pairwise features for all (i, j) particle pairs.
        
        particle_feats: (batch, N, 4) where features are (pT, eta, phi, flag)
        
        Returns:
        --------
        pair_bias: (batch, N, N) — attention bias from pairwise interactions
        """
        batch_size, n_parts, _ = particle_feats.shape
        
        # Extract individual features
        pt = particle_feats[:, :, 0]                     # (batch, N)
        eta = particle_feats[:, :, 1]                    # (batch, N)
        phi = particle_feats[:, :, 2]                    # (batch, N)
        
        # Compute pairwise differences
        # Δη_ij = η_i - η_j
        d_eta = eta.unsqueeze(2) - eta.unsqueeze(1)      # (batch, N, N)
        # Δφ_ij = φ_i - φ_j  
        d_phi = phi.unsqueeze(2) - phi.unsqueeze(1)      # (batch, N, N)
        # ΔR_ij = sqrt(Δη² + Δφ²)
        d_R = torch.sqrt(d_eta**2 + d_phi**2 + 1e-8)    # (batch, N, N)
        # pT ratio: min(pT_i, pT_j) / max(pT_i, pT_j)
        pt_i = pt.unsqueeze(2).expand(-1, -1, n_parts)   # (batch, N, N)
        pt_j = pt.unsqueeze(1).expand(-1, n_parts, -1)   # (batch, N, N)
        pt_min = torch.min(pt_i, pt_j)
        pt_max = torch.max(pt_i, pt_j).clamp(min=1e-8)
        pt_ratio = pt_min / pt_max                       # (batch, N, N)
        # kT = min(pT_i, pT_j) * ΔR_ij
        kT = pt_min * d_R                                # (batch, N, N)
        
        # Stack pairwise features: (batch, N, N, 5)
        pair_feats = torch.stack([d_eta, d_phi, d_R, pt_ratio, kT], dim=-1)
        
        # Project to attention bias: (batch, N, N, 5) → (batch, N, N)
        pair_bias = self.pair_proj(pair_feats).squeeze(-1)
        
        # Mask invalid pairs (where either particle is padded)
        if mask is not None:
            pair_mask = mask.unsqueeze(1) * mask.unsqueeze(2)  # (batch, N, N)
            pair_bias = pair_bias * pair_mask + (1 - pair_mask) * (-1e9)
        
        return pair_bias


class ParticleAttentionBlock(nn.Module):
    """
    Multi-head self-attention block with pairwise interaction bias.
    """
    def __init__(self, d_model, nhead, dim_feedforward=128, dropout=0.1):
        super(ParticleAttentionBlock, self).__init__()
        
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        
        # Q, K, V projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)           # Output projection
        
        # Layer norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feedforward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),                                   # GELU activation (smoother ReLU)
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
        
        self.attn_dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5               # Attention scaling factor
    
    def forward(self, x, pair_bias=None, mask=None):
        """
        x: (batch, N, d_model)
        pair_bias: (batch, N, N) — pairwise attention bias
        mask: (batch, N) — particle mask
        """
        batch_size, seq_len, _ = x.shape
        
        # --- Multi-head self-attention with pairwise bias ---
        residual = x
        x = self.norm1(x)                                # Pre-norm
        
        # Compute Q, K, V
        Q = self.W_q(x).reshape(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.W_k(x).reshape(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.W_v(x).reshape(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        # Q, K, V: (batch, nhead, N, head_dim)
        
        # Attention scores: Q · K^T / sqrt(d_k)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # (batch, nhead, N, N)
        
        # Add pairwise interaction bias (shared across heads)
        if pair_bias is not None:
            attn = attn + pair_bias.unsqueeze(1)         # Broadcast across heads
        
        # Mask attention for padded particles
        if mask is not None:
            attn_mask = mask.unsqueeze(1).unsqueeze(2)   # (batch, 1, 1, N)
            attn = attn.masked_fill(attn_mask == 0, -1e9)
        
        # Softmax and dropout
        attn = torch.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        
        # Apply attention to values
        out = torch.matmul(attn, V)                      # (batch, nhead, N, head_dim)
        out = out.transpose(1, 2).reshape(batch_size, seq_len, self.d_model)
        out = self.W_o(out)
        
        # Residual connection
        x = residual + out
        
        # --- Feedforward ---
        residual = x
        x = self.norm2(x)
        x = residual + self.ffn(x)
        
        return x


class ParticleTransformer(nn.Module):
    """
    Particle Transformer (ParT) — full model.
    
    Combines:
    - Per-particle input embedding
    - Pairwise interaction features as attention bias
    - Multiple transformer layers
    - Global pooling + classifier
    """
    def __init__(self, particle_dims=4, event_dims=7, d_model=64,
                 nhead=4, num_layers=3, dim_feedforward=128, dropout=0.1):
        super(ParticleTransformer, self).__init__()
        
        # Input embedding: project particle features to d_model
        self.input_embed = nn.Sequential(
            nn.Linear(particle_dims, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        
        # Pairwise interaction feature module
        self.pairwise = PairwiseFeatures(pair_feat_dim=5, d_model=32)
        
        # Transformer blocks with pairwise bias
        self.blocks = nn.ModuleList([
            ParticleAttentionBlock(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        
        # Class token (learnable)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        # Final layer norm
        self.final_norm = nn.LayerNorm(d_model)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(d_model + event_dims, 64),         # cls_token + event features
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, event_feats, particle_feats, mask):
        """
        event_feats: (batch, 7)
        particle_feats: (batch, N, 4)
        mask: (batch, N)
        """
        batch_size = particle_feats.size(0)
        
        # Compute pairwise interaction bias
        pair_bias = self.pairwise(particle_feats, mask)  # (batch, N, N)
        
        # Embed particles
        x = self.input_embed(particle_feats)             # (batch, N, d_model)
        
        # Prepend class token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # (batch, 1, d_model)
        x = torch.cat([cls_tokens, x], dim=1)            # (batch, N+1, d_model)
        
        # Extend mask and pair_bias for class token
        cls_mask = torch.ones(batch_size, 1)              # Class token always valid
        extended_mask = torch.cat([cls_mask, mask], dim=1)  # (batch, N+1)
        
        # Extend pairwise bias: class token attends to all, all attend to class token
        # Add row and column of zeros for cls token (no pairwise bias for cls)
        pair_bias_extended = torch.zeros(batch_size, x.size(1), x.size(1))
        pair_bias_extended[:, 1:, 1:] = pair_bias        # Original bias for particles
        
        # Apply transformer blocks
        for block in self.blocks:
            x = block(x, pair_bias_extended, extended_mask)
        
        # Extract class token representation
        x = self.final_norm(x)
        cls_output = x[:, 0, :]                          # (batch, d_model)
        
        # Combine with event features and classify
        combined = torch.cat([cls_output, event_feats], dim=1)
        output = self.classifier(combined)
        return output


# Instantiate
particle_transformer = ParticleTransformer(
    particle_dims=N_FEAT_PER_PARTICLE,
    event_dims=7,
    d_model=64,                  # Embedding dimension
    nhead=4,                     # Attention heads
    num_layers=3,                # Transformer layers
    dim_feedforward=128,         # FFN hidden size
    dropout=0.1
)
print(f"Particle Transformer parameters: {sum(p.numel() for p in particle_transformer.parameters()):,}")

# Train (lower LR for transformer)
history_part = train_model(
    particle_transformer, train_loader, val_loader, epochs=60, lr=0.0005
)

# Evaluate
y_prob_part, acc_part, auc_part = evaluate_model(
    particle_transformer, test_loader, y_test, "Particle Transformer"
)

In [ ]:
# ============================================================
# MODEL 5: LORENTZNET (Lorentz Equivariant GNN)
# ============================================================
# Reference: arXiv:2201.08187 (Gong et al., 2022)
#
# Key idea: The network respects Lorentz symmetry of spacetime.
# Uses both scalar features (Lorentz invariant) and coordinate features
# (Lorentz covariant) that transform properly under boosts/rotations.
#
# In our simplified version:
#   - Coordinates: (pT, eta, phi) — 3D kinematic space
#   - Scalars: learned embeddings of all particle features
#   - Message passing uses pairwise "distances" in coordinate space
#   - Coordinate updates are equivariant (relative displacements only)
#
# This is simplified from the full Lorentz-equivariant version
# (which operates on 4-vectors), adapted for (pT, eta, phi, flag) inputs.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""           # Force CPU

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

# ============================================================
# 1. LOAD AND PARSE DATA
# ============================================================

sig_df = pd.read_csv('test_data_signal.csv', header=None)
bkg_df = pd.read_csv('test_data_background.csv', header=None)

MAX_PARTICLES = 50
N_FEAT_PER_PARTICLE = 4        # pT, eta, phi, flag
OBJ_START = 8


def parse_data(df, max_particles=MAX_PARTICLES):
    """Parse raw CSV into structured arrays."""
    raw = df.values
    n_events = len(df)
    
    event_features = raw[:, 1:8].astype(np.float32)        # Columns 1-7
    particle_features = np.zeros(
        (n_events, max_particles, N_FEAT_PER_PARTICLE), dtype=np.float32
    )
    particle_mask = np.zeros((n_events, max_particles), dtype=np.float32)
    
    for i in range(n_events):
        n_obj = int(raw[i, 0])
        n_obj = min(n_obj, max_particles)
        for j in range(n_obj):
            col_start = OBJ_START + j * N_FEAT_PER_PARTICLE
            col_end = col_start + N_FEAT_PER_PARTICLE
            if col_end <= raw.shape[1]:
                particle_features[i, j, :] = raw[i, col_start:col_end]
                particle_mask[i, j] = 1.0
    
    return event_features, particle_features, particle_mask


# Parse
sig_evt, sig_parts, sig_mask = parse_data(sig_df)
bkg_evt, bkg_parts, bkg_mask = parse_data(bkg_df)

# Combine
event_features = np.concatenate([sig_evt, bkg_evt], axis=0)
particle_features = np.concatenate([sig_parts, bkg_parts], axis=0)
particle_mask = np.concatenate([sig_mask, bkg_mask], axis=0)
labels = np.concatenate([
    np.ones(len(sig_evt)), np.zeros(len(bkg_evt))
]).astype(np.float32)

print(f"Total events: {len(labels)}")
print(f"Event features: {event_features.shape}")
print(f"Particle features: {particle_features.shape}")

# Train/val/test split
(evt_train, evt_test,
 parts_train, parts_test,
 mask_train, mask_test,
 y_train, y_test) = train_test_split(
    event_features, particle_features, particle_mask, labels,
    test_size=0.2, random_state=42, stratify=labels
)

(evt_tr, evt_val,
 parts_tr, parts_val,
 mask_tr, mask_val,
 y_tr, y_val) = train_test_split(
    evt_train, parts_train, mask_train, y_train,
    test_size=0.15, random_state=42
)

print(f"Train: {len(y_tr)}, Val: {len(y_val)}, Test: {len(y_test)}")

# ============================================================
# 2. DATASET AND DATALOADER
# ============================================================

class EventDataset(Dataset):
    """Custom dataset for event data."""
    def __init__(self, evt_feats, part_feats, mask, labels):
        self.evt = torch.FloatTensor(evt_feats)
        self.parts = torch.FloatTensor(part_feats)
        self.mask = torch.FloatTensor(mask)
        self.labels = torch.FloatTensor(labels).unsqueeze(1)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.evt[idx], self.parts[idx], self.mask[idx], self.labels[idx]


BATCH_SIZE = 64
train_dataset = EventDataset(evt_tr, parts_tr, mask_tr, y_tr)
val_dataset = EventDataset(evt_val, parts_val, mask_val, y_val)
test_dataset = EventDataset(evt_test, parts_test, mask_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 3. LORENTZNET MODEL DEFINITION
# ============================================================


class LorentzNetLayer(nn.Module):
    """
    One layer of LorentzNet message passing.
    
    Updates both:
      - Scalar (invariant) features h_i — using pairwise Minkowski-like products
      - Coordinate features x_i — using learned equivariant displacements
    
    The key constraint: coordinate updates only use RELATIVE positions (x_j - x_i),
    ensuring equivariance under translations. The scalar features are updated
    using pairwise distances (invariant under translations/rotations).
    """
    def __init__(self, scalar_dim, coord_dim, hidden_dim=64):
        """
        Parameters:
        -----------
        scalar_dim : int
            Dimension of scalar (invariant) features per particle.
        coord_dim : int
            Dimension of coordinate features (3 for pT, eta, phi).
        hidden_dim : int
            Hidden dimension for internal MLPs.
        """
        super(LorentzNetLayer, self).__init__()
        
        self.scalar_dim = scalar_dim
        self.coord_dim = coord_dim
        
        # Message function: computes messages between particle pairs
        # Input: [h_i, h_j, d_ij^2] where d_ij is Euclidean distance in coord space
        # This is Lorentz-invariant because it only depends on distances
        self.message_mlp = nn.Sequential(
            nn.Linear(2 * scalar_dim + 1, hidden_dim),     # [h_i, h_j, dist²]
            nn.SiLU(),                                     # Smooth activation (SiLU = x*sigmoid(x))
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, scalar_dim)              # Output: message vector
        )
        
        # Coordinate update weight: determines how much each neighbor
        # pulls the particle in coordinate space
        # This ensures equivariance: update is a weighted sum of (x_j - x_i)
        self.coord_weight_mlp = nn.Sequential(
            nn.Linear(scalar_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),                      # Single scalar weight
            nn.Tanh()                                      # Bounded [-1, 1]
        )
        
        # Node update: combines old features with aggregated messages
        self.node_update = nn.Sequential(
            nn.Linear(2 * scalar_dim, hidden_dim),         # [h_i, aggregated_msg]
            nn.SiLU(),
            nn.Linear(hidden_dim, scalar_dim)
        )
        
        # Layer normalization for stability
        self.layer_norm = nn.LayerNorm(scalar_dim)
    
    def forward(self, h, x, mask):
        """
        Parameters:
        -----------
        h : (batch, N, scalar_dim) — scalar features per particle
        x : (batch, N, coord_dim) — coordinate features per particle
        mask : (batch, N) — particle mask (1=real, 0=padding)
        
        Returns:
        --------
        h_new : (batch, N, scalar_dim) — updated scalar features
        x_new : (batch, N, coord_dim) — updated coordinates
        """
        batch_size, n_parts, _ = h.shape
        
        # --- Compute pairwise distances in coordinate space ---
        # dx_ij = x_j - x_i (relative displacement)
        # Shape: (batch, N, N, coord_dim)
        dx = x.unsqueeze(1) - x.unsqueeze(2)              # (batch, N_i, N_j, coord_dim)
        
        # Squared Euclidean distance: d²_ij = ||x_i - x_j||²
        # This is invariant under translations
        d_sq = (dx ** 2).sum(dim=-1, keepdim=True)         # (batch, N, N, 1)
        
        # --- Compute messages ---
        # For each pair (i, j): message = MLP([h_i, h_j, d²_ij])
        h_i = h.unsqueeze(2).expand(-1, -1, n_parts, -1)  # (batch, N, N, scalar_dim)
        h_j = h.unsqueeze(1).expand(-1, n_parts, -1, -1)  # (batch, N, N, scalar_dim)
        
        # Concatenate: [h_i, h_j, d²_ij]
        msg_input = torch.cat([h_i, h_j, d_sq], dim=-1)   # (batch, N, N, 2S+1)
        
        # Apply message MLP
        messages = self.message_mlp(msg_input)             # (batch, N, N, scalar_dim)
        
        # --- Apply pair mask ---
        # Only consider messages between real particles
        pair_mask = mask.unsqueeze(1) * mask.unsqueeze(2)  # (batch, N, N)
        # Zero out self-interactions (i == j)
        eye = torch.eye(n_parts).unsqueeze(0)              # (1, N, N)
        pair_mask = pair_mask * (1 - eye)                  # Remove diagonal
        
        messages = messages * pair_mask.unsqueeze(-1)      # Zero padded pairs
        
        # --- Aggregate messages ---
        # Sum messages from all neighbors, normalize by neighbor count
        n_neighbors = pair_mask.sum(dim=-1, keepdim=True).clamp(min=1)  # (batch, N, 1)
        agg_messages = messages.sum(dim=2) / n_neighbors   # (batch, N, scalar_dim)
        
        # --- Update scalar features (residual connection) ---
        h_input = torch.cat([h, agg_messages], dim=-1)     # (batch, N, 2*scalar_dim)
        h_update = self.node_update(h_input)               # (batch, N, scalar_dim)
        h_new = h + h_update                               # Residual
        h_new = self.layer_norm(h_new)                     # Normalize
        
        # --- Update coordinates (equivariant update) ---
        # The coordinate update is a weighted sum of displacement vectors:
        # Δx_i = Σ_j w_ij * (x_j - x_i)
        # where w_ij is learned from the messages
        # This preserves equivariance because it's built from relative positions
        
        coord_weights = self.coord_weight_mlp(messages)    # (batch, N, N, 1)
        coord_weights = coord_weights * pair_mask.unsqueeze(-1)  # Mask
        
        # Weighted sum of displacements: Σ_j w_ij * (x_j - x_i)
        # dx: (batch, N, N, coord_dim), coord_weights: (batch, N, N, 1)
        x_shift = (coord_weights * dx).sum(dim=2)          # (batch, N, coord_dim)
        x_shift = x_shift / n_neighbors                    # Normalize
        
        x_new = x + 0.1 * x_shift                         # Small step (stability)
        
        # --- Apply particle mask ---
        h_new = h_new * mask.unsqueeze(-1)
        x_new = x_new * mask.unsqueeze(-1)
        
        return h_new, x_new


class LorentzNet(nn.Module):
    """
    LorentzNet: Lorentz-equivariant graph neural network for particle classification.
    
    Architecture:
      1. Embed particle features into scalar space
      2. Separate coordinates (pT, eta, phi) for equivariant processing
      3. Apply multiple LorentzNet layers (message passing + coord update)
      4. Pool over particles (invariant aggregation)
      5. Combine with event features → classify
    
    The model naturally handles variable-length inputs via masking.
    """
    def __init__(self, particle_dims=4, event_dims=7, scalar_dim=64,
                 n_layers=4, hidden_dim=64):
        """
        Parameters:
        -----------
        particle_dims : int
            Number of raw features per particle (4: pT, eta, phi, flag).
        event_dims : int
            Number of event-level features (7).
        scalar_dim : int
            Dimension of learned scalar representations.
        n_layers : int
            Number of LorentzNet message passing layers.
        hidden_dim : int
            Hidden dimension for MLPs within each layer.
        """
        super(LorentzNet, self).__init__()
        
        self.coord_dim = 3                                 # Use (pT, eta, phi) as coordinates
        self.scalar_dim = scalar_dim
        self.n_layers = n_layers
        
        # Initial scalar feature embedding
        # Maps ALL particle features (pT, eta, phi, flag) → scalar space
        # These scalar features will be updated by message passing
        self.scalar_embed = nn.Sequential(
            nn.Linear(particle_dims, scalar_dim),
            nn.SiLU(),
            nn.Linear(scalar_dim, scalar_dim),
            nn.LayerNorm(scalar_dim)
        )
        
        # Optional: initial coordinate transform (learnable preprocessing)
        # Helps normalize the coordinate space
        self.coord_embed = nn.Sequential(
            nn.Linear(3, 3),                               # Linear transform of (pT, eta, phi)
        )
        
        # Stack of LorentzNet layers
        self.layers = nn.ModuleList([
            LorentzNetLayer(scalar_dim, self.coord_dim, hidden_dim)
            for _ in range(n_layers)
        ])
        
        # Pooling: learn attention weights for weighted sum (better than mean)
        self.attention_pool = nn.Sequential(
            nn.Linear(scalar_dim, 32),
            nn.Tanh(),
            nn.Linear(32, 1)                               # Attention score per particle
        )
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(scalar_dim + event_dims, 128),       # Pooled + event features
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.SiLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()                                   # Binary probability
        )
    
    def forward(self, event_feats, particle_feats, mask):
        """
        Parameters:
        -----------
        event_feats : (batch, 7) — event-level features
        particle_feats : (batch, N, 4) — per-particle (pT, eta, phi, flag)
        mask : (batch, N) — particle mask
        
        Returns:
        --------
        (batch, 1) — signal probability
        """
        # --- Separate coordinates and embed scalars ---
        # Coordinates: (pT, eta, phi)
        x = particle_feats[:, :, :3]                       # (batch, N, 3)
        x = self.coord_embed(x)                            # Optional learned transform
        x = x * mask.unsqueeze(-1)                         # Mask padding
        
        # Scalar features: embed all particle features
        h = self.scalar_embed(particle_feats)              # (batch, N, scalar_dim)
        h = h * mask.unsqueeze(-1)                         # Mask padding
        
        # --- Apply LorentzNet layers ---
        for layer in self.layers:
            h, x = layer(h, x, mask)                       # Update both h and x
        
        # --- Invariant pooling ---
        # Attention-weighted sum over particles
        # This is translation/permutation invariant
        attn_scores = self.attention_pool(h)               # (batch, N, 1)
        attn_scores = attn_scores + (1 - mask.unsqueeze(-1)) * (-1e9)  # Mask padding
        attn_weights = torch.softmax(attn_scores, dim=1)  # (batch, N, 1)
        attn_weights = attn_weights * mask.unsqueeze(-1)   # Zero out padding
        
        # Weighted sum
        pooled = (attn_weights * h).sum(dim=1)             # (batch, scalar_dim)
        
        # --- Classify ---
        combined = torch.cat([pooled, event_feats], dim=1) # (batch, scalar_dim + 7)
        output = self.classifier(combined)                 # (batch, 1)
        
        return output


# ============================================================
# 4. INSTANTIATE MODEL
# ============================================================

print("=" * 60)
print("MODEL: LORENTZNET (Simplified Lorentz-Equivariant GNN)")
print("=" * 60)

lorentznet_model = LorentzNet(
    particle_dims=N_FEAT_PER_PARTICLE,         # 4: pT, eta, phi, flag
    event_dims=7,                              # 7 event-level features
    scalar_dim=64,                             # Scalar feature dimension
    n_layers=4,                                # 4 message-passing layers
    hidden_dim=64                              # Hidden MLP dimension
).to(DEVICE)

# Print model info
total_params = sum(p.numel() for p in lorentznet_model.parameters())
trainable_params = sum(p.numel() for p in lorentznet_model.parameters() if p.requires_grad)
print(f"\n  Scalar dimension: 64")
print(f"  Coordinate dimension: 3 (pT, eta, phi)")
print(f"  Message-passing layers: 4")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# ============================================================
# 5. TRAINING LOOP
# ============================================================

print("\n" + "=" * 60)
print("TRAINING LORENTZNET")
print("=" * 60)

criterion = nn.BCELoss()                                   # Binary cross-entropy loss
optimizer = optim.Adam(
    lorentznet_model.parameters(),
    lr=0.0005,                                             # Lower LR for stability
    weight_decay=1e-5                                      # L2 regularization
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=7          # Halve LR after 7 stale epochs
)

EPOCHS = 60
PATIENCE = 15
best_val_loss = float('inf')
patience_counter = 0
best_state = None

history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

for epoch in range(EPOCHS):
    # --- Training phase ---
    lorentznet_model.train()
    train_loss = 0.0
    n_train_samples = 0
    
    for evt, parts, mask, lbl in train_loader:
        optimizer.zero_grad()                              # Clear gradients
        
        output = lorentznet_model(evt, parts, mask)        # Forward pass
        loss = criterion(output, lbl)                      # Compute loss
        
        loss.backward()                                    # Backpropagation
        
        # Gradient clipping (important for GNNs — prevents exploding gradients)
        torch.nn.utils.clip_grad_norm_(lorentznet_model.parameters(), max_norm=1.0)
        
        optimizer.step()                                   # Update weights
        
        train_loss += loss.item() * evt.size(0)            # Accumulate
        n_train_samples += evt.size(0)
    
    train_loss /= n_train_samples                          # Average over epoch
    history['train_loss'].append(train_loss)
    
    # --- Validation phase ---
    lorentznet_model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels_list = []
    n_val_samples = 0
    
    with torch.no_grad():                                  # No gradient computation
        for evt, parts, mask, lbl in val_loader:
            output = lorentznet_model(evt, parts, mask)
            loss = criterion(output, lbl)
            val_loss += loss.item() * evt.size(0)
            val_preds.append(output.numpy())
            val_labels_list.append(lbl.numpy())
            n_val_samples += evt.size(0)
    
    val_loss /= n_val_samples
    val_preds_arr = np.concatenate(val_preds).ravel()
    val_labels_arr = np.concatenate(val_labels_list).ravel()
    val_auc = roc_auc_score(val_labels_arr, val_preds_arr)
    
    history['val_loss'].append(val_loss)
    history['val_auc'].append(val_auc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model weights
        best_state = {k: v.clone() for k, v in lorentznet_model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping triggered at epoch {epoch+1}")
            break
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}: "
              f"train_loss={train_loss:.4f}, "
              f"val_loss={val_loss:.4f}, "
              f"val_auc={val_auc:.4f}, "
              f"lr={current_lr:.6f}")

# Restore best model weights
lorentznet_model.load_state_dict(best_state)
print(f"\nTraining complete. Best validation loss: {best_val_loss:.4f}")

# ============================================================
# 6. EVALUATION ON TEST SET
# ============================================================

print("\n" + "=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

lorentznet_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for evt, parts, mask, lbl in test_loader:
        output = lorentznet_model(evt, parts, mask)
        all_preds.append(output.numpy())
        all_labels.append(lbl.numpy())

y_prob = np.concatenate(all_preds).ravel()                 # Predicted probabilities
y_pred = (y_prob >= 0.5).astype(int)                       # Hard predictions at 0.5 threshold

# Metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nLorentzNet Results:")
print(f"  Accuracy:  {acc:.4f}")
print(f"  AUC-ROC:   {auc:.4f}")
print(f"  Parameters: {total_params:,}")

# Confusion at different working points
print(f"\nWorking points:")
for threshold in [0.3, 0.5, 0.7, 0.9]:
    preds_wp = (y_prob >= threshold).astype(int)
    sig_eff = preds_wp[y_test == 1].mean()                 # Signal efficiency
    bkg_rej = 1.0 - preds_wp[y_test == 0].mean()          # Background rejection
    print(f"  Threshold={threshold:.1f}: sig_eff={sig_eff:.4f}, bkg_rej={bkg_rej:.4f}")

# ============================================================
# 7. ROC CURVE
# ============================================================

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Standard ROC curve
ax1.plot(fpr, tpr, color='darkgreen', lw=2, label=f'LorentzNet (AUC = {auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
ax1.set_xlabel('False Positive Rate (Background Efficiency)')
ax1.set_ylabel('True Positive Rate (Signal Efficiency)')
ax1.set_title('ROC Curve — LorentzNet')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Signal efficiency vs background rejection (log scale)
# More useful for physics: how much background do you reject at a given signal efficiency?
bkg_rejection = 1.0 / (fpr + 1e-10)                       # 1/FPR
valid = fpr > 0                                            # Avoid division by zero
ax2.plot(tpr[valid], bkg_rejection[valid], color='darkgreen', lw=2)
ax2.set_xlabel('Signal Efficiency (TPR)')
ax2.set_ylabel('Background Rejection (1/FPR)')
ax2.set_yscale('log')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([1, 1e4])
ax2.set_title('Signal Efficiency vs Background Rejection')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('roc_lorentznet.pdf')
plt.show()

# ============================================================
# 8. TRAINING CURVES
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
ax1.plot(history['train_loss'], label='Train Loss', color='blue')
ax1.plot(history['val_loss'], label='Val Loss', color='orange')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Binary Cross-Entropy Loss')
ax1.set_title('LorentzNet Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# AUC curve
ax2.plot(history['val_auc'], label='Val AUC', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('AUC-ROC')
ax2.set_title('LorentzNet Validation AUC')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lorentznet_training_curves.pdf')
plt.show()

# ============================================================
# 9. VISUALIZE LEARNED COORDINATE EVOLUTION
# ============================================================
# Show how particle coordinates evolve through the layers

print("\n" + "=" * 60)
print("COORDINATE EVOLUTION THROUGH LAYERS")
print("=" * 60)

# Take one example event and trace coordinates through layers
lorentznet_model.eval()

# Pick a signal event and a background event
sig_idx = np.where(y_test == 1)[0][0]                      # First signal event in test
bkg_idx = np.where(y_test == 0)[0][0]                      # First background event in test

def get_coordinate_evolution(model, evt, parts, mask):
    """
    Run forward pass and collect coordinates after each layer.
    Returns list of coordinate arrays: [(N, 3), (N, 3), ...]
    """
    model.eval()
    with torch.no_grad():
        # Add batch dimension
        evt = torch.FloatTensor(evt).unsqueeze(0)
        parts = torch.FloatTensor(parts).unsqueeze(0)
        mask = torch.FloatTensor(mask).unsqueeze(0)
        
        # Initial coordinates
        x = model.coord_embed(parts[:, :, :3])
        x = x * mask.unsqueeze(-1)
        h = model.scalar_embed(parts)
        h = h * mask.unsqueeze(-1)
        
        coords_history = [x.squeeze(0).numpy().copy()]     # Layer 0 (input)
        
        # Forward through layers, saving coordinates
        for layer in model.layers:
            h, x = layer(h, x, mask)
            coords_history.append(x.squeeze(0).numpy().copy())
    
    return coords_history


# Get coordinate evolution for signal and background events
sig_coords = get_coordinate_evolution(
    lorentznet_model,
    evt_test[sig_idx], parts_test[sig_idx], mask_test[sig_idx]
)
bkg_coords = get_coordinate_evolution(
    lorentznet_model,
    evt_test[bkg_idx], parts_test[bkg_idx], mask_test[bkg_idx]
)

# Plot eta-phi plane evolution
n_layers_plot = len(sig_coords)
fig, axes = plt.subplots(2, n_layers_plot, figsize=(4*n_layers_plot, 8))

sig_n_parts = int(mask_test[sig_idx].sum())
bkg_n_parts = int(mask_test[bkg_idx].sum())

for layer_idx in range(n_layers_plot):
    # Signal event (top row)
    ax = axes[0, layer_idx]
    coords = sig_coords[layer_idx][:sig_n_parts]           # Only real particles
    pt_sizes = parts_test[sig_idx, :sig_n_parts, 0]        # pT for marker size
    sizes = 20 + 100 * (pt_sizes / pt_sizes.max())         # Scale marker size by pT
    ax.scatter(coords[:, 1], coords[:, 2], s=sizes, alpha=0.6, c='blue')
    ax.set_title(f'Layer {layer_idx}' if layer_idx > 0 else 'Input')
    if layer_idx == 0:
        ax.set_ylabel('Signal\n(coord dim 2)')
    ax.set_xlabel('coord dim 1')
    ax.grid(True, alpha=0.2)
    
    # Background event (bottom row)
    ax = axes[1, layer_idx]
    coords = bkg_coords[layer_idx][:bkg_n_parts]
    pt_sizes = parts_test[bkg_idx, :bkg_n_parts, 0]
    sizes = 20 + 100 * (pt_sizes / pt_sizes.max())
    ax.scatter(coords[:, 1], coords[:, 2], s=sizes, alpha=0.6, c='red')
    if layer_idx == 0:
        ax.set_ylabel('Background\n(coord dim 2)')
    ax.set_xlabel('coord dim 1')
    ax.grid(True, alpha=0.2)

plt.suptitle('LorentzNet: Particle Coordinate Evolution Through Layers\n'
             '(marker size ∝ pT)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('lorentznet_coord_evolution.pdf', bbox_inches='tight')
plt.show()

# ============================================================
# 10. ATTENTION POOLING WEIGHTS ANALYSIS
# ============================================================
# Examine which particles the model pays most attention to

print("\n" + "=" * 60)
print("ATTENTION WEIGHT ANALYSIS")
print("=" * 60)


def get_attention_weights(model, evt, parts, mask):
    """Extract attention pooling weights for one event."""
    model.eval()
    with torch.no_grad():
        evt_t = torch.FloatTensor(evt).unsqueeze(0)
        parts_t = torch.FloatTensor(parts).unsqueeze(0)
        mask_t = torch.FloatTensor(mask).unsqueeze(0)
        
        # Run through layers
        x = model.coord_embed(parts_t[:, :, :3]) * mask_t.unsqueeze(-1)
        h = model.scalar_embed(parts_t) * mask_t.unsqueeze(-1)
        for layer in model.layers:
            h, x = layer(h, x, mask_t)
        
        # Get attention scores
        attn_scores = model.attention_pool(h)
        attn_scores = attn_scores + (1 - mask_t.unsqueeze(-1)) * (-1e9)
        attn_weights = torch.softmax(attn_scores, dim=1)
        attn_weights = attn_weights * mask_t.unsqueeze(-1)
    
    return attn_weights.squeeze().numpy()


# Compute attention weights for many signal and background events
n_analyze = min(200, int(y_test.sum()), int((1-y_test).sum()))

sig_attns = []
bkg_attns = []
sig_pts_all = []
bkg_pts_all = []

sig_indices = np.where(y_test == 1)[0][:n_analyze]
bkg_indices = np.where(y_test == 0)[0][:n_analyze]

for idx in sig_indices:
    attn = get_attention_weights(
        lorentznet_model, evt_test[idx], parts_test[idx], mask_test[idx]
    )
    n_real = int(mask_test[idx].sum())
    sig_attns.extend(attn[:n_real].tolist())
    sig_pts_all.extend(parts_test[idx, :n_real, 0].tolist())

for idx in bkg_indices:
    attn = get_attention_weights(
        lorentznet_model, evt_test[idx], parts_test[idx], mask_test[idx]
    )
    n_real = int(mask_test[idx].sum())
    bkg_attns.extend(attn[:n_real].tolist())
    bkg_pts_all.extend(parts_test[idx, :n_real, 0].tolist())

sig_attns = np.array(sig_attns)
bkg_attns = np.array(bkg_attns)
sig_pts_all = np.array(sig_pts_all)
bkg_pts_all = np.array(bkg_pts_all)

# Plot attention vs pT
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: attention weight vs pT
ax1.scatter(sig_pts_all[:500], sig_attns[:500], alpha=0.3, s=10, c='blue', label='Signal')
ax1.scatter(bkg_pts_all[:500], bkg_attns[:500], alpha=0.3, s=10, c='red', label='Background')
ax1.set_xlabel('Particle pT')
ax1.set_ylabel('Attention Weight')
ax1.set_title('Attention Weight vs Particle pT')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Histogram of attention weights
ax2.hist(sig_attns, bins=50, alpha=0.5, density=True, label='Signal', color='blue')
ax2.hist(bkg_attns, bins=50, alpha=0.5, density=True, label='Background', color='red')
ax2.set_xlabel('Attention Weight')
ax2.set_ylabel('Density')
ax2.set_title('Distribution of Attention Weights')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lorentznet_attention_analysis.pdf')
plt.show()

# ============================================================
# 11. OUTPUT SCORE DISTRIBUTION
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

# Separate predictions by true class
sig_scores = y_prob[y_test == 1]
bkg_scores = y_prob[y_test == 0]

ax.hist(sig_scores, bins=50, alpha=0.6, density=True, label='Signal', color='blue')
ax.hist(bkg_scores, bins=50, alpha=0.6, density=True, label='Background', color='red')
ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold = 0.5')
ax.set_xlabel('LorentzNet Output Score')
ax.set_ylabel('Normalized Distribution')
ax.set_title('LorentzNet Discriminant Output')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lorentznet_output_distribution.pdf')
plt.show()

print("\nDone! All plots saved.")